# ACDC Cardiac MRI — DiceCE + InstanceNorm + Augmentation

**Author:** Ali Al Balushi (3539318) · Deep Learning in Medical Image Analysis 2026  
**Team:** Joan Prenafeta Ribau · Ali Al Balushi · Fabian Krüger · Samsensurya Selvaraj

**This notebook is locked to the `DiceCE_INSTANCE_AUG` experiment.**

What this notebook does:
- Preprocesses the ACDC dataset (V2: per-volume clip+zscore, fixed seed split)
- Trains 2D U-Net with DiceCE loss + InstanceNorm + data augmentation
- Evaluates Dice and Hausdorff per structure (LV, RV, MYO)
- Applies post-processing (largest connected component + hole filling)
- Reports ED vs ES split metrics
- Compares results against Bernard et al. 2018

How to run:
1. Open in Google Colab
2. Runtime → Change runtime type → T4 GPU
3. Run all cells in order

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q monai nibabel
print('ready')

In [ ]:
from __future__ import annotations
import os, re, json, random, time
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.ndimage import zoom
from tqdm import tqdm
import torch
import monai
from monai.transforms import (
    Compose, EnsureChannelFirstd, Lambdad,
    NormalizeIntensityd, SpatialPadd, CenterSpatialCropd, ToTensord,
    RandFlipd, RandRotate90d,
)
from monai.data import Dataset, DataLoader
from monai.networks.nets import UNet
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.networks.utils import one_hot

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device} | monai: {monai.__version__} | torch: {torch.__version__}')

## 1) Experiment Config

This notebook is fixed to `DiceCE_INSTANCE_AUG`. Do not change these settings.

In [ ]:
# ── FIXED CONFIGURATION — DiceCE + InstanceNorm + Augmentation ───────────────
LOSS_NAME = 'DiceCE'
NORM_UNIT  = 'INSTANCE'
EXP_NAME   = 'DiceCE_INSTANCE_AUG'

ACDC_ROOT  = Path('/content/drive/MyDrive/ACDC-2')
RAW_SPLIT  = 'training'
OUT_2D     = ACDC_ROOT / 'database' / 'prepared_2d_v2'
SPLIT_JSON = OUT_2D / 'patient_split.json'

MODEL_DIR  = ACDC_ROOT / 'models' / EXP_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH  = MODEL_DIR / 'last.pt'

VAL_RATIO         = 0.2
SEED              = 42
TARGET_SPACING_XY = (1.37, 1.37)
TARGET_SIZE       = (256, 256)
CLIP_LOW, CLIP_HIGH = 1, 99
BATCH_SIZE  = 4
NUM_WORKERS = 2
MAX_EPOCHS          = 80
EARLY_STOP_PATIENCE = 15

RAW_TRAIN_DIR = ACDC_ROOT / 'database' / RAW_SPLIT
patients = sorted([p for p in RAW_TRAIN_DIR.iterdir() if p.is_dir()])
print(f'Experiment   : {EXP_NAME}')
print(f'Loss         : {LOSS_NAME}')
print(f'Norm unit    : {NORM_UNIT}')
print(f'Augmentation : RandFlip (H+V, p=0.5) + RandRotate90 (p=0.5)')
print(f'Model dir    : {MODEL_DIR}')
print(f'Patients     : {len(patients)}')

## 2) Export 2D slices — V2 preprocessing

Only needs to run once. If `prepared_2d_v2/` already has your slices, skip this section.

V2 changes over V1:
- Per-volume clip(1-99) + z-score instead of per-slice min-max
- Fixed patient-level split saved to JSON (no data leakage)

In [ ]:
def normalize_volume_v2(vol, clip_low=1, clip_high=99):
    """
    V2 normalization: clip outliers across the whole volume, then z-score.
    Better than V1 per-slice min-max which was sensitive to outlier pixels.
    """
    vol = vol.astype(np.float32)
    vol = np.clip(vol, np.percentile(vol, clip_low), np.percentile(vol, clip_high))
    std = vol.std()
    return ((vol - vol.mean()) / std).astype(np.float32) if std > 1e-8 else np.zeros_like(vol)


def resample_volume_xy(img, msk, spacing, target=(1.37, 1.37)):
    """Resample x/y to common spacing."""
    zx, zy = spacing[0]/target[0], spacing[1]/target[1]
    return (zoom(img, (zx, zy, 1.0), order=1).astype(np.float32),
            zoom(msk, (zx, zy, 1.0), order=0).astype(np.uint8))


def create_split(raw_dir, out_json, val_ratio=0.2, seed=42):
    """Create 80/20 patient-level split and save to JSON."""
    out_json = Path(out_json)
    out_json.parent.mkdir(parents=True, exist_ok=True)
    if out_json.exists():
        split = json.load(open(out_json))
        print(f'loaded existing split: train={len(split["train"])} | val={len(split["val"])}')
        return split
    pts = sorted([p.name for p in Path(raw_dir).iterdir()
                  if p.is_dir() and p.name.startswith('patient')])
    rng = random.Random(seed); rng.shuffle(pts)
    n = max(1, int(round(len(pts) * val_ratio)))
    split = {'train': pts[n:], 'val': pts[:n], 'seed': seed}
    json.dump(split, open(out_json, 'w'), indent=2)
    print(f'split saved: train={len(split["train"])} | val={len(split["val"])}')
    return split


split = create_split(RAW_TRAIN_DIR, SPLIT_JSON, VAL_RATIO, SEED)

In [ ]:
def export_patient(patient_name, split_name):
    pdir    = RAW_TRAIN_DIR / patient_name
    img_out = OUT_2D / split_name / 'images'
    msk_out = OUT_2D / split_name / 'masks'
    img_out.mkdir(parents=True, exist_ok=True)
    msk_out.mkdir(parents=True, exist_ok=True)
    nii_files = sorted(pdir.glob(f'{pdir.name}_frame*.nii.gz'))
    img_files = [f for f in nii_files if not f.name.endswith('_gt.nii.gz')]
    written = 0
    for img_path in img_files:
        msk_path = img_path.with_name(img_path.name.replace('.nii.gz', '_gt.nii.gz'))
        if not msk_path.exists(): continue
        img_nii  = nib.load(str(img_path))
        img_vol  = img_nii.get_fdata().astype(np.float32)
        msk_vol  = nib.load(str(msk_path)).get_fdata().astype(np.uint8)
        spacing  = img_nii.header.get_zooms()[:3]
        img_vol, msk_vol = resample_volume_xy(img_vol, msk_vol, spacing, TARGET_SPACING_XY)
        img_vol = normalize_volume_v2(img_vol, CLIP_LOW, CLIP_HIGH)
        for k in range(img_vol.shape[-1]):
            name = img_path.name.replace('.nii.gz', f'_slice{k:03d}.npy')
            np.save(img_out / name, img_vol[..., k])
            np.save(msk_out / name, msk_vol[..., k])
            written += 1
    return written


n_existing = len(list((OUT_2D / 'train' / 'images').glob('*.npy')))
if n_existing > 0:
    print(f'slices already exported ({n_existing} train slices found) -- skipping export')
else:
    total_tr = sum(export_patient(n, 'train') for n in tqdm(split['train'], desc='train'))
    total_vl = sum(export_patient(n, 'val')   for n in tqdm(split['val'],   desc='val'))
    print(f'done — train: {total_tr} | val: {total_vl} slices')

## 3) DataLoaders + Augmentation

Augmentation is applied **only during training** (not validation or inference):
- Random horizontal flip (p=0.5)
- Random vertical flip (p=0.5)
- Random 90° rotation (p=0.5, k ∈ {1,2,3})

In [ ]:
def load_img(p): return np.load(p).astype(np.float32)
def load_msk(p): return np.load(p).astype(np.uint8)


# Training transform: includes augmentation
train_tfm = Compose([
    Lambdad(keys=['img'],  func=lambda p: load_img(p)),
    Lambdad(keys=['mask'], func=lambda p: load_msk(p)),
    EnsureChannelFirstd(keys=['img', 'mask'], channel_dim='no_channel'),
    NormalizeIntensityd(keys=['img'], nonzero=False, channel_wise=True),
    SpatialPadd(keys=['img', 'mask'], spatial_size=TARGET_SIZE),
    CenterSpatialCropd(keys=['img', 'mask'], roi_size=TARGET_SIZE),
    # ── AUGMENTATION (enabled for DiceCE_INSTANCE_AUG) ──
    RandFlipd(keys=['img', 'mask'], prob=0.5, spatial_axis=0),
    RandFlipd(keys=['img', 'mask'], prob=0.5, spatial_axis=1),
    RandRotate90d(keys=['img', 'mask'], prob=0.5, max_k=3),
    ToTensord(keys=['img', 'mask']),
])

# Validation transform: NO augmentation
val_tfm = Compose([
    Lambdad(keys=['img'],  func=lambda p: load_img(p)),
    Lambdad(keys=['mask'], func=lambda p: load_msk(p)),
    EnsureChannelFirstd(keys=['img', 'mask'], channel_dim='no_channel'),
    NormalizeIntensityd(keys=['img'], nonzero=False, channel_wise=True),
    SpatialPadd(keys=['img', 'mask'], spatial_size=TARGET_SIZE),
    CenterSpatialCropd(keys=['img', 'mask'], roi_size=TARGET_SIZE),
    ToTensord(keys=['img', 'mask']),
])


def make_dicts(split_name):
    img_dir = OUT_2D / split_name / 'images'
    msk_dir = OUT_2D / split_name / 'masks'
    msk_map = {p.name: p for p in sorted(msk_dir.glob('*.npy'))}
    return [{'img': str(p), 'mask': str(msk_map[p.name])}
            for p in sorted(img_dir.glob('*.npy')) if p.name in msk_map]


train_dicts = make_dicts('train')
val_dicts   = make_dicts('val')
train_loader = DataLoader(Dataset(train_dicts, train_tfm), batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(Dataset(val_dicts,   val_tfm),   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)

b = next(iter(train_loader))
print(f'train: {len(train_dicts)} slices | val: {len(val_dicts)} slices')
print(f'img shape: {b["img"].shape} | labels: {torch.unique(b["mask"])}')
print('Augmentation: RandFlip(H) + RandFlip(V) + RandRotate90 — ENABLED')

## 4) V1 vs V2 Normalization Comparison

In [ ]:
%matplotlib inline

example_patient = sorted([p for p in RAW_TRAIN_DIR.iterdir() if p.is_dir()])[0]
example_img = sorted([f for f in example_patient.glob('*.nii.gz')
                       if not f.name.endswith('_gt.nii.gz')])[0]

raw_vol = nib.load(str(example_img)).get_fdata().astype(np.float32)
if raw_vol.ndim == 4: raw_vol = raw_vol[:, :, :, 0]
mid = raw_vol.shape[-1] // 2
raw_slice = raw_vol[:, :, mid]

v1 = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)
v2 = normalize_volume_v2(raw_vol)[:, :, mid]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(raw_slice, cmap='gray')
axes[0].set_title(f'Raw [{raw_slice.min():.0f}, {raw_slice.max():.0f}]'); axes[0].axis('off')
axes[1].imshow(v1, cmap='gray')
axes[1].set_title(f'V1: per-slice min-max\n[{v1.min():.2f}, {v1.max():.2f}]'); axes[1].axis('off')
axes[2].imshow(v2, cmap='gray')
axes[2].set_title(f'V2: volume clip+zscore\nmean={v2.mean():.2f} std={v2.std():.2f}'); axes[2].axis('off')
plt.suptitle(f'{example_patient.name} — V2 normalization more robust to outliers')
plt.tight_layout()
plt.savefig(str(OUT_2D / 'v1_vs_v2_normalization.png'), dpi=150, bbox_inches='tight')
plt.show()
print('saved: v1_vs_v2_normalization.png')

## 5) Training

**Option A — Train from scratch:** run 5a and 5b.  
**Option B — Load existing checkpoint:** skip 5a/5b, run 5c directly.

In [ ]:
# ── 5a) Model + loss ─────────────────────────────────────────────────────────
model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=4,
    channels=(32, 64, 128, 256, 256),
    strides=(2, 2, 2, 2),
    num_res_units=1,
    act=('LEAKYRELU', {'negative_slope': 0.01, 'inplace': True}),
    norm=(NORM_UNIT, {}),
    dropout=0.0,
).to(device)

loss_fn     = monai.losses.DiceCELoss(to_onehot_y=True, softmax=True)
optimizer   = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-6)
dice_metric = monai.metrics.DiceMetric(include_background=False, reduction='mean')

print(f'model    : 2D U-Net ({NORM_UNIT})')
print(f'loss     : {LOSS_NAME} -> {loss_fn.__class__.__name__}')
print(f'aug      : RandFlip(H+V) + RandRotate90')
print(f'saving to: {MODEL_DIR}')

In [ ]:
# ── 5b) Training loop ────────────────────────────────────────────────────────
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train(); total = 0
    for batch in loader:
        imgs  = batch['img'].to(device)
        masks = batch['mask'].to(device).long().squeeze(1)
        optimizer.zero_grad(set_to_none=True)
        logits = model(imgs)
        loss = loss_fn(logits, masks.unsqueeze(1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / max(1, len(loader))


@torch.no_grad()
def val_one_epoch(model, loader, loss_fn, dice_metric, device):
    model.eval(); total = 0; dice_metric.reset()
    for batch in loader:
        imgs  = batch['img'].to(device)
        masks = batch['mask'].to(device).long().squeeze(1)
        logits = model(imgs)
        total += loss_fn(logits, masks.unsqueeze(1)).item()
        pred_oh = monai.networks.one_hot(
            torch.argmax(torch.softmax(logits, 1), 1).unsqueeze(1), 4)
        gt_oh = monai.networks.one_hot(masks.unsqueeze(1), 4)
        dice_metric(y_pred=pred_oh, y=gt_oh)
    return total / max(1, len(loader)), dice_metric.aggregate().item()


best_dice  = -1.0
no_improve = 0
history    = []

print(f'training [{EXP_NAME}] ...')

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    tr_loss          = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    vl_loss, vl_dice = val_one_epoch(model, val_loader, loss_fn, dice_metric, device)
    scheduler.step(vl_loss)
    lr = optimizer.param_groups[0]['lr']

    history.append({'epoch': epoch, 'train_loss': tr_loss,
                    'val_loss': vl_loss, 'val_dice': vl_dice, 'lr': lr})

    print(f'Epoch {epoch:03d}/{MAX_EPOCHS} | '
          f'train={tr_loss:.4f} | val={vl_loss:.4f} | '
          f'dice={vl_dice:.4f} | lr={lr:.1e} | {time.time()-t0:.0f}s')

    ckpt = {'epoch': epoch, 'exp_name': EXP_NAME, 'loss_name': LOSS_NAME,
            'norm_unit': NORM_UNIT, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_dice': best_dice}
    torch.save(ckpt, MODEL_DIR / 'last.pt')

    if vl_dice > best_dice:
        best_dice  = vl_dice; no_improve = 0
        torch.save(ckpt, MODEL_DIR / 'best.pt')
        print(f'  -> new best: {best_dice:.4f} saved best.pt')
    else:
        no_improve += 1

    pd.DataFrame(history).to_csv(MODEL_DIR / 'history.csv', index=False)

    if no_improve >= EARLY_STOP_PATIENCE:
        print(f'early stopping at epoch {epoch}')
        break

print(f'done [{EXP_NAME}] | best val dice: {best_dice:.4f}')

In [ ]:
# ── 5c) Load checkpoint ──────────────────────────────────────────────────────
ckpt_file = MODEL_DIR / 'best.pt' if (MODEL_DIR / 'best.pt').exists() else CKPT_PATH

model = UNet(
    spatial_dims=2, in_channels=1, out_channels=4,
    channels=(32, 64, 128, 256, 256), strides=(2, 2, 2, 2),
    num_res_units=1,
    act=('LEAKYRELU', {'negative_slope': 0.01, 'inplace': True}),
    norm=(NORM_UNIT, {}),
    dropout=0.0,
).to(device)

ckpt = torch.load(str(ckpt_file), map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'loaded   : {ckpt_file}')
print(f'epoch    : {ckpt.get("epoch")}')
print(f'best dice: {ckpt.get("best_dice"):.4f}')
print(f'exp      : {ckpt.get("exp_name", EXP_NAME)}')

In [ ]:
# ── 5d) Training curves ───────────────────────────────────────────────────────
%matplotlib inline

history = pd.read_csv(MODEL_DIR / 'history.csv')

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(history['epoch'], history['train_loss'], label='Train loss')
ax[0].plot(history['epoch'], history['val_loss'],   label='Val loss')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss')
ax[0].set_title(f'Loss curves [{EXP_NAME}]'); ax[0].legend(); ax[0].grid(True)
ax[1].plot(history['epoch'], history['val_dice'], label='Val Dice', color='green')
ax[1].axhline(history['val_dice'].max(), color='red', linestyle='--',
              label=f'Best: {history["val_dice"].max():.4f}')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Dice')
ax[1].set_title(f'Validation Dice [{EXP_NAME}]'); ax[1].legend(); ax[1].grid(True)
plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Best val Dice: {history["val_dice"].max():.4f} at epoch {history["val_dice"].idxmax()+1}')
print(f'saved: {MODEL_DIR}/training_curves.png')

## 6) Qualitative predictions

GT overlay | Prediction overlay | Prediction + GT contours

In [ ]:
%matplotlib inline

CLASS_COLORS = {1: np.array([1.,0.,0.]), 2: np.array([0.,1.,0.]), 3: np.array([0.,0.,1.])}

def make_overlay(img, mask, alpha=0.35):
    base = img.astype(np.float32); mn, mx = base.min(), base.max()
    base = (base-mn)/(mx-mn+1e-8)
    rgb  = np.stack([base]*3, axis=-1)
    for cls, col in CLASS_COLORS.items():
        r = (mask==cls)
        if r.any(): rgb[r] = (1-alpha)*rgb[r] + alpha*col
    return np.clip(rgb, 0, 1)


@torch.no_grad()
def show_predictions(model, loader, device, n=6):
    model.eval(); examples = []
    for batch in loader:
        imgs  = batch['img'].to(device)
        masks = batch['mask'].to(device).long()
        preds = torch.argmax(model(imgs), dim=1)
        for i in range(imgs.shape[0]):
            examples.append((imgs[i,0].cpu().numpy(),
                             masks[i,0].cpu().numpy(),
                             preds[i].cpu().numpy()))
    examples = random.sample(examples, min(n, len(examples)))
    fig, axes = plt.subplots(n, 3, figsize=(15, 5*n))
    if n == 1: axes = axes[None]
    for r, (img, gt, pred) in enumerate(examples):
        axes[r,0].imshow(make_overlay(img,gt));   axes[r,0].set_title('GT overlay');         axes[r,0].axis('off')
        axes[r,1].imshow(make_overlay(img,pred)); axes[r,1].set_title('Prediction overlay'); axes[r,1].axis('off')
        axes[r,2].imshow(make_overlay(img,pred))
        for cls, col in CLASS_COLORS.items():
            b = (gt==cls).astype(np.uint8)
            if b.sum()>0: axes[r,2].contour(b, levels=[0.5], colors=[col], linewidths=1.5)
        axes[r,2].set_title('Pred + GT contours'); axes[r,2].axis('off')
    plt.tight_layout()
    plt.savefig(str(MODEL_DIR / 'predictions.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'saved: {MODEL_DIR}/predictions.png')


show_predictions(model, val_loader, device, n=6)

## 7) 3D Inference

Fabian's fix: resample softmax probabilities back to original grid before argmax.  
V1 resampled hard labels which caused boundary artefacts and large Hausdorff distances.

In [ ]:
def zscore_slice(img):
    img = img.astype(np.float32); sd = img.std()
    return (img-img.mean())/sd if sd>1e-8 else np.zeros_like(img)

def pad_crop(img, size):
    th, tw = size; h, w = img.shape
    ph, pw = max(0,th-h), max(0,tw-w)
    if ph or pw: img = np.pad(img, ((ph//2,ph-ph//2),(pw//2,pw-pw//2)))
    h, w = img.shape
    return img[(h-th)//2:(h-th)//2+th, (w-tw)//2:(w-tw)//2+tw]

def load_nii(p): nimg=nib.load(str(p)); return nimg.get_fdata(), nimg.affine, nimg.header
def save_nii(p, data, affine, header): nib.save(nib.Nifti1Image(data,affine,header), str(p))


@torch.no_grad()
def predict_volume(model, vol, spacing, device, txy=TARGET_SPACING_XY, tsz=TARGET_SIZE):
    model.eval(); H, W, Z = vol.shape; sx, sy = spacing[0], spacing[1]
    slices = []
    for k in range(Z):
        sl = vol[:,:,k].astype(np.float32)
        sl_rs = zoom(sl, (sx/txy[0], sy/txy[1]), order=1)
        x = torch.from_numpy(pad_crop(zscore_slice(sl_rs), tsz)).float()[None,None].to(device)
        probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
        probs_orig = np.zeros((4,H,W), dtype=np.float32)
        for c in range(4):
            p = zoom(probs[c], (H/probs.shape[1], W/probs.shape[2]), order=1)
            hh,ww = min(H,p.shape[0]), min(W,p.shape[1])
            probs_orig[c,:hh,:ww] = p[:hh,:ww]
        slices.append(np.argmax(probs_orig, axis=0).astype(np.uint8))
    return np.stack(slices, axis=2)


PRED_DIR = ACDC_ROOT / 'eval_outputs' / EXP_NAME
PRED_DIR.mkdir(parents=True, exist_ok=True)

for pdir in tqdm(sorted([p for p in RAW_TRAIN_DIR.iterdir() if p.is_dir()])):
    out = PRED_DIR / pdir.name; out.mkdir(parents=True, exist_ok=True)
    for img_path in sorted([p for p in pdir.glob(f'{pdir.name}_frame*.nii.gz')
                             if not p.name.endswith('_gt.nii.gz')]):
        nii  = nib.load(str(img_path))
        pred = predict_volume(model, nii.get_fdata().astype(np.float32),
                              nii.header.get_zooms()[:3], device)
        save_nii(out/img_path.name, pred.astype(np.uint8), nii.affine, nii.header)

print(f'inference done [{EXP_NAME}] -> {PRED_DIR}')

## 8) Segmentation metrics — Dice + Hausdorff

In [ ]:
HEADER = ['Name','Dice LV','HD LV (mm)','Volume LV (ml)','Err LV (ml)',
          'Dice RV','HD RV (mm)','Volume RV (ml)','Err RV (ml)',
          'Dice MYO','HD MYO (mm)','Volume MYO (ml)','Err MYO (ml)']

def vol_ml(mask, cls, vs): return ((mask==cls).sum()*np.prod(vs))/1000.0

def onehot3d(m, nc=4):
    return one_hot(torch.from_numpy(m.astype(np.int64))[None,None], num_classes=nc).float()

def seg_metrics(gt, pred, vs):
    vs = tuple(float(x) for x in vs)
    dm = DiceMetric(include_background=False, reduction='none', ignore_empty=False)
    hm = HausdorffDistanceMetric(include_background=False, percentile=100.0,
                                  directed=False, reduction='none')
    dm.reset(); hm.reset()
    dm(y_pred=onehot3d(pred), y=onehot3d(gt))
    hm(y_pred=onehot3d(pred), y=onehot3d(gt), spacing=vs)
    d=dm.aggregate().cpu().numpy()[0]; h=hm.aggregate().cpu().numpy()[0]
    geom={'RV':(d[0],h[0]),'MYO':(d[1],h[1]),'LV':(d[2],h[2])}
    vl_p=vol_ml(pred,3,vs);vl_g=vol_ml(gt,3,vs)
    vr_p=vol_ml(pred,1,vs);vr_g=vol_ml(gt,1,vs)
    vm_p=vol_ml(pred,2,vs);vm_g=vol_ml(gt,2,vs)
    return [geom['LV'][0],geom['LV'][1],vl_p,vl_p-vl_g,
            geom['RV'][0],geom['RV'][1],vr_p,vr_p-vr_g,
            geom['MYO'][0],geom['MYO'][1],vm_p,vm_p-vm_g]

def compute_seg_df(gt_dir, pred_dir):
    gt_files = sorted(Path(gt_dir).rglob('*_gt.nii.gz'))
    pred_map = {p.name:p for p in sorted(Path(pred_dir).rglob('*.nii.gz'))}
    rows=[]
    for gp in gt_files:
        pn=gp.name.replace('_gt.nii.gz','.nii.gz')
        if pn not in pred_map: continue
        gt,_,hdr=load_nii(gp); pred,_,_=load_nii(pred_map[pn])
        rows.append(pd.DataFrame([[gp.name.split('.')[0]]+
                    seg_metrics(gt,pred,hdr.get_zooms()[:3])], columns=HEADER))
    return pd.concat(rows, ignore_index=True)

df_seg = compute_seg_df(RAW_TRAIN_DIR, PRED_DIR)
print(f'Results [{EXP_NAME}]:')
display(df_seg.drop(columns=['Name']).mean(numeric_only=True).to_frame('mean').T)

In [ ]:
%matplotlib inline

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
dice_m={s:df_seg[f'Dice {s}'].mean() for s in ['LV','RV','MYO']}
hd_m  ={s:df_seg[f'HD {s} (mm)'].mean() for s in ['LV','RV','MYO']}
bars0 = ax[0].bar(dice_m.keys(), dice_m.values(), color=['steelblue','coral','mediumseagreen'])
ax[0].set_title(f'Mean Dice [{EXP_NAME}]'); ax[0].set_ylim(0,1); ax[0].grid(axis='y')
for bar, val in zip(bars0, dice_m.values()):
    ax[0].text(bar.get_x()+bar.get_width()/2, val+0.005, f'{val:.3f}', ha='center', fontsize=9)
bars1 = ax[1].bar(hd_m.keys(), hd_m.values(), color=['steelblue','coral','mediumseagreen'])
ax[1].set_title(f'Mean Hausdorff [{EXP_NAME}] (mm)'); ax[1].grid(axis='y')
for bar, val in zip(bars1, hd_m.values()):
    ax[1].text(bar.get_x()+bar.get_width()/2, val+0.3, f'{val:.1f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(str(MODEL_DIR/'3D_Dice_HD.png'), dpi=150, bbox_inches='tight')
plt.show(); print(f'saved: {MODEL_DIR}/3D_Dice_HD.png')

## 9) Clinical metrics

LVEDV, LVEF, RVEDV, RVEF, MYMass with BSA normalization (Bernard et al. 2018).

In [ ]:
def myo_mass(mask,vs,rho=1.05): return vol_ml(mask,2,vs)*rho
def ef(edv,esv): return 100.*(edv-esv)/edv if edv>0 else np.nan
def parse_frame(name):
    m=re.search(r'(patient\d+)_frame(\d+)',name); return m.group(1),m.group(2)
def read_cfg(path):
    info={}
    for line in open(path,'r'):
        if ':' in line.strip(): k,v=line.strip().split(':',1); info[k.strip()]=v.strip()
    return info
def bsa(w,h): return 0.007184*(w**0.425)*(h**0.725) if w>0 and h>0 else np.nan
def corr_bias_std_mae(gt,pred):
    gt=np.asarray(gt,float); pred=np.asarray(pred,float); err=pred-gt
    return (np.corrcoef(gt,pred)[0,1] if len(gt)>1 else np.nan),\
           np.mean(err),(np.std(err,ddof=1) if len(err)>1 else 0.),np.mean(np.abs(err))

def clinical_table(gt_dir,pred_dir):
    gt_files=sorted(Path(gt_dir).rglob('*_gt.nii.gz'))
    pred_map={p.name:p for p in sorted(Path(pred_dir).rglob('*.nii.gz'))}
    rows=[]
    for gp in gt_files:
        pn=gp.name.replace('_gt.nii.gz','.nii.gz')
        if pn not in pred_map: continue
        gt,_,hdr=load_nii(gp); pred,_,_=load_nii(pred_map[pn])
        vs=tuple(float(x) for x in hdr.get_zooms()[:3])
        patient,frame=parse_frame(gp.name)
        cfg=read_cfg(gp.parent/'Info.cfg')
        b=bsa(float(cfg['Weight']),float(cfg['Height']))
        rows.append({'patient':patient,'frame_int':int(frame),
            'is_ED':int(frame)==int(cfg['ED']),'is_ES':int(frame)==int(cfg['ES']),'BSA':b,
            'LV_gt':vol_ml(gt,3,vs)/b,'LV_pred':vol_ml(pred,3,vs)/b,
            'RV_gt':vol_ml(gt,1,vs)/b,'RV_pred':vol_ml(pred,1,vs)/b,
            'MY_gt':myo_mass(gt,vs)/b,'MY_pred':myo_mass(pred,vs)/b})
    return pd.DataFrame(rows)

def summarize(df):
    rows=[]
    for pat,g in df.groupby('patient'):
        ed=g[g['is_ED']]; es=g[g['is_ES']]
        if len(ed)!=1 or len(es)!=1: continue
        ed=ed.iloc[0]; es=es.iloc[0]
        rows.append({'patient':pat,
            'LVEDV_gt':ed['LV_gt'],'LVEDV_pred':ed['LV_pred'],
            'LVEF_gt':ef(ed['LV_gt'],es['LV_gt']),'LVEF_pred':ef(ed['LV_pred'],es['LV_pred']),
            'RVEDV_gt':ed['RV_gt'],'RVEDV_pred':ed['RV_pred'],
            'RVEF_gt':ef(ed['RV_gt'],es['RV_gt']),'RVEF_pred':ef(ed['RV_pred'],es['RV_pred']),
            'MYMass_gt':ed['MY_gt'],'MYMass_pred':ed['MY_pred']})
    return pd.DataFrame(rows)

def clin_summary(df_pat):
    pairs=[('LVEDV','LVEDV_gt','LVEDV_pred'),('LVEF','LVEF_gt','LVEF_pred'),
           ('RVEDV','RVEDV_gt','RVEDV_pred'),('RVEF','RVEF_gt','RVEF_pred'),
           ('MYMass','MYMass_gt','MYMass_pred')]
    return pd.DataFrame([{'Metric':n,
        'corr':corr_bias_std_mae(df_pat[g],df_pat[p])[0],
        'bias':corr_bias_std_mae(df_pat[g],df_pat[p])[1],
        'std':corr_bias_std_mae(df_pat[g],df_pat[p])[2],
        'mae':corr_bias_std_mae(df_pat[g],df_pat[p])[3]} for n,g,p in pairs])

df_cf=clinical_table(RAW_TRAIN_DIR, PRED_DIR)
df_cp=summarize(df_cf)
df_cs=clin_summary(df_cp)
print(f'Clinical summary [{EXP_NAME}]:')
display(df_cs)

In [ ]:
%matplotlib inline

pairs=[('LVEDV_gt','LVEDV_pred','LVEDV (ml/m²)'),('LVEF_gt','LVEF_pred','LVEF (%)'),
       ('RVEDV_gt','RVEDV_pred','RVEDV (ml/m²)'),('RVEF_gt','RVEF_pred','RVEF (%)'),
       ('MYMass_gt','MYMass_pred','MY Mass (g/m²)')]
fig,axes=plt.subplots(2,3,figsize=(14,8)); axes=axes.flatten()
for i,(g,p,t) in enumerate(pairs):
    x=df_cp[g]; y=df_cp[p]
    axes[i].scatter(x,y,alpha=0.6); mn=min(x.min(),y.min()); mx=max(x.max(),y.max())
    axes[i].plot([mn,mx],[mn,mx],'--',color='gray')
    axes[i].set_xlabel('GT'); axes[i].set_ylabel('Pred'); axes[i].set_title(t)
axes[-1].axis('off')
plt.suptitle(f'Clinical metrics [{EXP_NAME}]')
plt.tight_layout()
plt.savefig(str(MODEL_DIR/'clinical_metrics.png'), dpi=150, bbox_inches='tight')
plt.show(); print(f'saved: {MODEL_DIR}/clinical_metrics.png')

## 9b) Post-processing — Largest Connected Component + Hole Filling

Cleans segmentation masks by:
1. Keeping only the largest connected component per class (removes disconnected blobs)
2. Filling holes inside each structure (morphological closing slice-by-slice)

This does **not** require retraining — applied directly to saved NIfTI predictions.

In [ ]:
from scipy.ndimage import label, binary_fill_holes

def keep_largest_component(binary_mask):
    if binary_mask.sum() == 0: return binary_mask
    labeled, n = label(binary_mask)
    if n == 1: return binary_mask
    sizes = [(labeled == i).sum() for i in range(1, n + 1)]
    largest = np.argmax(sizes) + 1
    return (labeled == largest).astype(np.uint8)

def fill_holes_3d(binary_mask):
    filled = np.zeros_like(binary_mask)
    for k in range(binary_mask.shape[2]):
        filled[:, :, k] = binary_fill_holes(binary_mask[:, :, k]).astype(np.uint8)
    return filled

def postprocess_mask(mask):
    result = np.zeros_like(mask)
    for cls in [1, 2, 3]:  # RV, MYO, LV
        binary = (mask == cls).astype(np.uint8)
        binary = keep_largest_component(binary)
        binary = fill_holes_3d(binary)
        result[binary == 1] = cls
    return result


PRED_DIR_RAW = ACDC_ROOT / 'eval_outputs' / EXP_NAME
PRED_DIR_PP  = ACDC_ROOT / 'eval_outputs' / f'{EXP_NAME}_PP'
PRED_DIR_PP.mkdir(parents=True, exist_ok=True)

processed = 0
for pdir in tqdm(sorted([p for p in PRED_DIR_RAW.iterdir() if p.is_dir()])):
    out = PRED_DIR_PP / pdir.name; out.mkdir(parents=True, exist_ok=True)
    for pred_path in sorted(pdir.glob('*.nii.gz')):
        pred, affine, header = load_nii(pred_path)
        pred_pp = postprocess_mask(pred.astype(np.uint8))
        save_nii(out / pred_path.name, pred_pp.astype(np.uint8), affine, header)
        processed += 1

print(f'post-processing done: {processed} volumes')
print(f'saved to: {PRED_DIR_PP}')

In [ ]:
%matplotlib inline

df_raw = compute_seg_df(RAW_TRAIN_DIR, PRED_DIR_RAW)
df_pp  = compute_seg_df(RAW_TRAIN_DIR, PRED_DIR_PP)

print('RAW predictions:')
display(df_raw.drop(columns=['Name']).mean(numeric_only=True).to_frame('mean').T)
print('POST-PROCESSED predictions:')
display(df_pp.drop(columns=['Name']).mean(numeric_only=True).to_frame('mean').T)

structs = ['LV', 'RV', 'MYO']
dice_raw = [df_raw[f'Dice {s}'].mean() for s in structs]
dice_pp  = [df_pp[f'Dice {s}'].mean()  for s in structs]
hd_raw   = [df_raw[f'HD {s} (mm)'].mean() for s in structs]
hd_pp    = [df_pp[f'HD {s} (mm)'].mean()  for s in structs]

x = np.arange(len(structs)); w = 0.35
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].bar(x - w/2, dice_raw, w, label='Raw',            color='steelblue')
ax[0].bar(x + w/2, dice_pp,  w, label='Post-processed', color='darkorange')
ax[0].set_xticks(x); ax[0].set_xticklabels(structs)
ax[0].set_title('Dice — Raw vs Post-processed'); ax[0].set_ylim(0, 1); ax[0].legend(); ax[0].grid(axis='y')
for i,(r,p) in enumerate(zip(dice_raw,dice_pp)):
    ax[0].text(i-w/2, r+0.005, f'{r:.3f}', ha='center', fontsize=8)
    ax[0].text(i+w/2, p+0.005, f'{p:.3f}', ha='center', fontsize=8)
ax[1].bar(x - w/2, hd_raw, w, label='Raw',            color='steelblue')
ax[1].bar(x + w/2, hd_pp,  w, label='Post-processed', color='darkorange')
ax[1].set_xticks(x); ax[1].set_xticklabels(structs)
ax[1].set_title('Hausdorff (mm) — Raw vs Post-processed'); ax[1].legend(); ax[1].grid(axis='y')
for i,(r,p) in enumerate(zip(hd_raw,hd_pp)):
    ax[1].text(i-w/2, r+0.3, f'{r:.1f}', ha='center', fontsize=8)
    ax[1].text(i+w/2, p+0.3, f'{p:.1f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'postprocessing_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('saved: postprocessing_comparison.png')

## 9c) ED/ES Split Evaluation

Reports Dice and Hausdorff separately for End Diastole (ED) and End Systole (ES).  
Required for direct comparison with Bernard et al. 2018 Table III.

In [ ]:
%matplotlib inline

def compute_seg_df_by_phase(gt_dir, pred_dir):
    gt_files = sorted(Path(gt_dir).rglob('*_gt.nii.gz'))
    pred_map = {p.name: p for p in sorted(Path(pred_dir).rglob('*.nii.gz'))}
    rows = []
    for gp in gt_files:
        pn = gp.name.replace('_gt.nii.gz', '.nii.gz')
        if pn not in pred_map: continue
        gt, _, hdr = load_nii(gp); pred, _, _ = load_nii(pred_map[pn])
        cfg = {}
        cfg_path = gp.parent / 'Info.cfg'
        if cfg_path.exists():
            for line in open(cfg_path):
                if ':' in line: k,v=line.strip().split(':',1); cfg[k.strip()]=v.strip()
        patient, frame = re.search(r'(patient\d+)_frame(\d+)', gp.name).groups()
        phase = 'ED' if int(frame)==int(cfg.get('ED','-1')) else \
                'ES' if int(frame)==int(cfg.get('ES','-1')) else 'other'
        row = [gp.name.split('.')[0]] + seg_metrics(gt, pred, hdr.get_zooms()[:3])
        df_row = pd.DataFrame([row], columns=HEADER)
        df_row['phase'] = phase
        df_row['patient'] = patient
        rows.append(df_row)
    return pd.concat(rows, ignore_index=True)


df_phase = compute_seg_df_by_phase(RAW_TRAIN_DIR, PRED_DIR_PP)

print(f'ED/ES split results [{EXP_NAME}]:')
for phase in ['ED', 'ES']:
    subset = df_phase[df_phase['phase'] == phase]
    print(f'\n--- {phase} ({len(subset)} frames) ---')
    means = subset[['Dice LV','HD LV (mm)','Dice RV','HD RV (mm)','Dice MYO','HD MYO (mm)']].mean()
    print(means.to_string())

x = np.arange(3); w = 0.35
ed = df_phase[df_phase['phase']=='ED']
es = df_phase[df_phase['phase']=='ES']

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].bar(x-w/2, [ed[f'Dice {s}'].mean() for s in ['LV','RV','MYO']], w, label='ED', color='steelblue')
ax[0].bar(x+w/2, [es[f'Dice {s}'].mean() for s in ['LV','RV','MYO']], w, label='ES', color='coral')
ax[0].set_xticks(x); ax[0].set_xticklabels(['LV','RV','MYO'])
ax[0].set_title('Dice by phase (post-processed)'); ax[0].set_ylim(0,1); ax[0].legend(); ax[0].grid(axis='y')
ax[1].bar(x-w/2, [ed[f'HD {s} (mm)'].mean() for s in ['LV','RV','MYO']], w, label='ED', color='steelblue')
ax[1].bar(x+w/2, [es[f'HD {s} (mm)'].mean() for s in ['LV','RV','MYO']], w, label='ES', color='coral')
ax[1].set_xticks(x); ax[1].set_xticklabels(['LV','RV','MYO'])
ax[1].set_title('Hausdorff by phase (mm)'); ax[1].legend(); ax[1].grid(axis='y')
plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'ED_ES_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('saved: ED_ES_comparison.png')

## 9d) Comparison vs Bernard et al. 2018

Bernard et al. 2018 (Table III) report the best-performing method on the ACDC test set.  
We compare our validation set results against those reference numbers.

**Fill in your actual results in `our_results` below after running sections 9b/9c.**

In [ ]:
%matplotlib inline

# ── Bernard et al. 2018 best method (Table III) ───────────────────────────────
# Source: Bernard et al., IEEE TMI 2018, doi:10.1109/TMI.2018.2837502
bernard = {
    'LV_ED': 0.940, 'LV_ES': 0.923,
    'RV_ED': 0.930, 'RV_ES': 0.882,
    'MYO_ED': 0.883, 'MYO_ES': 0.888,
}

# ── Pull our results from the phase dataframe automatically ───────────────────
ed = df_phase[df_phase['phase']=='ED']
es = df_phase[df_phase['phase']=='ES']

ours = {
    'LV_ED':  ed['Dice LV'].mean(),  'LV_ES':  es['Dice LV'].mean(),
    'RV_ED':  ed['Dice RV'].mean(),  'RV_ES':  es['Dice RV'].mean(),
    'MYO_ED': ed['Dice MYO'].mean(), 'MYO_ES': es['Dice MYO'].mean(),
}

# ── Build comparison table ────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Structure': ['LV (ED)', 'LV (ES)', 'RV (ED)', 'RV (ES)', 'MYO (ED)', 'MYO (ES)'],
    'Bernard et al. 2018': [bernard['LV_ED'], bernard['LV_ES'],
                             bernard['RV_ED'], bernard['RV_ES'],
                             bernard['MYO_ED'], bernard['MYO_ES']],
    f'Ours ({EXP_NAME})':  [ours['LV_ED'], ours['LV_ES'],
                             ours['RV_ED'], ours['RV_ES'],
                             ours['MYO_ED'], ours['MYO_ES']],
})
comparison['Diff'] = comparison[f'Ours ({EXP_NAME})'] - comparison['Bernard et al. 2018']
print('Dice comparison vs Bernard et al. 2018:')
display(comparison.round(3))

# ── Bar chart comparison ──────────────────────────────────────────────────────
structs_labels = ['LV\n(ED)', 'LV\n(ES)', 'RV\n(ED)', 'RV\n(ES)', 'MYO\n(ED)', 'MYO\n(ES)']
ref_vals  = list(comparison['Bernard et al. 2018'])
ours_vals = list(comparison[f'Ours ({EXP_NAME})'])

x = np.arange(len(structs_labels)); w = 0.35
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w/2, ref_vals,  w, label='Bernard et al. 2018', color='gray',       alpha=0.8)
ax.bar(x + w/2, ours_vals, w, label=f'Ours ({EXP_NAME})', color='steelblue',   alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(structs_labels)
ax.set_ylim(0, 1.05); ax.set_ylabel('Dice score')
ax.set_title('Dice comparison — Our model vs Bernard et al. 2018')
ax.legend(); ax.grid(axis='y')
for i, (r, o) in enumerate(zip(ref_vals, ours_vals)):
    ax.text(i-w/2, r+0.005, f'{r:.3f}', ha='center', fontsize=7.5)
    ax.text(i+w/2, o+0.005, f'{o:.3f}', ha='center', fontsize=7.5)
plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'bernard_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('saved: bernard_comparison.png')

## 10) Experiment comparison

Compares all completed experiments side by side.

In [ ]:
%matplotlib inline

exp_dirs = sorted([d for d in (ACDC_ROOT / 'models').iterdir()
                   if d.is_dir() and (d / 'history.csv').exists()])

if not exp_dirs:
    print('No experiments found yet.')
else:
    histories  = {}
    best_dices = {}
    for d in exp_dirs:
        h = pd.read_csv(d / 'history.csv')
        histories[d.name]  = h
        best_dices[d.name] = h['val_dice'].max()
        print(f'{d.name:25s} -> best val dice: {best_dices[d.name]:.4f} '
              f'(epoch {int(h.loc[h["val_dice"].idxmax(),"epoch"])})')

In [ ]:
%matplotlib inline

if histories:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for name, h in histories.items():
        lw = 2.5 if name == EXP_NAME else 1.0
        axes[0].plot(h['epoch'], h['val_dice'],
                     label=f'{name} (best={best_dices[name]:.4f})', linewidth=lw)
        axes[1].plot(h['epoch'], h['val_loss'], label=name, linewidth=lw)
    axes[0].set_title('Validation Dice — all experiments')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Dice')
    axes[0].legend(fontsize=8); axes[0].grid(True)
    axes[1].set_title('Validation Loss — all experiments')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(fontsize=8); axes[1].grid(True)
    plt.tight_layout()
    plt.savefig(str(OUT_2D / 'experiment_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('saved: experiment_comparison.png')
    print(f'\n{"Experiment":25s} {"Best Val Dice":>15s} {"Best Epoch":>12s}')
    print('-' * 55)
    for name, h in sorted(histories.items(), key=lambda x: -best_dices[x[0]]):
        best_ep = int(h.loc[h['val_dice'].idxmax(), 'epoch'])
        marker  = ' <-- best' if best_dices[name] == max(best_dices.values()) else ''
        print(f'{name:25s} {best_dices[name]:>15.4f} {best_ep:>12d}{marker}')

## 11) Summary

In [ ]:
with open(SPLIT_JSON) as f: s=json.load(f)
n_tr=len(list((OUT_2D/'train'/'images').glob('*.npy')))
n_vl=len(list((OUT_2D/'val'/'images').glob('*.npy')))

print('='*60)
print(f'EXPERIMENT: {EXP_NAME}')
print('='*60)
print(f'Loss function      : {LOSS_NAME} (DiceCE)')
print(f'Norm unit          : {NORM_UNIT} (InstanceNorm)')
print(f'Augmentation       : RandFlip(H+V, p=0.5) + RandRotate90(p=0.5)')
print(f'Train patients     : {len(s["train"])} ({n_tr} slices)')
print(f'Val patients       : {len(s["val"])} ({n_vl} slices)')
print(f'Spacing            : {TARGET_SPACING_XY} mm')
print(f'Size               : {TARGET_SIZE}')
print(f'Normalization      : clip({CLIP_LOW},{CLIP_HIGH}) + z-score (per volume)')
print(f'Seed               : {SEED}')
print('='*60)
if (MODEL_DIR/'history.csv').exists():
    h = pd.read_csv(MODEL_DIR/'history.csv')
    print(f'Best val Dice      : {h["val_dice"].max():.4f}')
    print(f'Best epoch         : {int(h.loc[h["val_dice"].idxmax(),"epoch"])}')
    print(f'Epochs trained     : {len(h)}')
print('='*60)
print('Outputs saved to:', MODEL_DIR)
for f in sorted(MODEL_DIR.glob('*.png')): print(f'  {f.name}')